# Noise-aware dynamic data pruning

Choose a GPU runtime. Run setup, then the experiment sections you need. Completed runs with matching settings are skipped. An interrupted training run restarts from epoch 0. Full experiments use 200 epochs and persistent Drive outputs.

Prepare a ZIP containing `requirements.txt`, `experiments/`, `infobatch_repro/`, `utils/`, `tests/`, `make_report.py`, and `make_figures.py`. A containing project folder is supported. Upload it when setup asks. Source files are kept in the Colab runtime, and new result files go to Drive.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile
from google.colab import drive, files

drive.mount('/content/drive')
source_dir = Path('/content/noise_aware_pruning')
source_dir.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
for filename in uploaded:
    if not zipfile.is_zipfile(filename):
        raise ValueError('Upload a ZIP of the project source.')
    with zipfile.ZipFile(filename) as archive:
        archive.extractall(source_dir)
roots = sorted({p.parent.parent for p in source_dir.rglob('experiments/feature_filters.py')})
if len(roots) != 1:
    raise ValueError('The ZIP must contain one project with experiments/feature_filters.py.')
os.chdir(roots[0])
if not Path('requirements.txt').is_file():
    raise FileNotFoundError('Include requirements.txt in the source ZIP.')
output = Path('/content/drive/MyDrive/noise_aware_pruning_outputs')
output.mkdir(parents=True, exist_ok=True)
os.environ['INFOBATCH_OUTPUT_DIR'] = str(output)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)

def run(script, *args):
    subprocess.run([sys.executable, script, *map(str, args)], check=True)

print('Source directory', Path.cwd())
print('Persistent results', output)


## Headline symmetric comparison

Three matched training seeds with fixed synthetic label corruption. This reproduces the main symmetric 40% noise comparison in the report.


In [ ]:
for seed in (0, 1, 2):
    run('experiments/feature_filters.py', '--noise', 'sym', '--eta', 0.4,
        '--runs', 'ib', 'ib_cen_iter', '--seed', seed)


## Baselines across the three noise settings

Full training is included here for symmetric noise. The missing asymmetric full-training run is in an optional section below. Yang-style and AlignPrune use the adaptations described in the report.


In [ ]:
for noise, eta in (('sym', 0.2), ('sym', 0.4), ('asym', 0.4)):
    methods = ['ib', 'ib_oracle', 'ib_centroid', 'ib_cen_iter', 'ib_clip', 'yang', 'yang_sel', 'ib_align']
    if noise == 'sym':
        methods.insert(0, 'full')
    run('experiments/feature_filters.py', '--noise', noise, '--eta', eta,
        '--runs', *methods, '--seed', 0)


## Additional seeds used in the report


In [ ]:
for seed in (1, 2):
    run('experiments/feature_filters.py', '--noise', 'sym', '--eta', 0.4,
        '--runs', 'ib_centroid', '--seed', seed)
    run('experiments/feature_filters.py', '--noise', 'asym', '--eta', 0.4,
        '--runs', 'ib_cen_iter', 'ib_clip', '--seed', seed)


## Signal diagnostics

The current diagnostic computes all ten signals. The `_v2` suffix is compatible with the figure script and the historical expanded diagnostics.


In [ ]:
for noise, eta in (('sym', 0.2), ('sym', 0.4), ('asym', 0.4)):
    run('experiments/uncertainty_signals.py', '--noise', noise, '--eta', eta, '--tag', 'v2')


## Early-filter and kNN ablations


In [ ]:
run('experiments/feature_filters.py', '--noise', 'sym', '--eta', 0.4,
    '--runs', 'ib_centroid', 'ib_knn', '--f0', 10, '--seed', 0)
run('experiments/feature_filters.py', '--noise', 'sym', '--eta', 0.4,
    '--runs', 'ib_knn', '--f0', 40, '--seed', 0)


## Clean-data reproduction


In [ ]:
run('infobatch_repro/repro.py')


## Optional missing baseline

This measures a result marked pending in the current report. Rebuild the tables after completion and update the narrative and main LaTeX table before presenting the new result.


In [ ]:
run('experiments/feature_filters.py', '--noise', 'asym', '--eta', 0.4, '--runs', 'full', '--seed', 0)


## Regression checks, tables, and figures

Generated results and plots go to the selected Drive output directory. The LaTeX narrative and main table require manual review before a new PDF is compiled.


In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
run('make_report.py')
run('make_figures.py')
print('Tables', output / 'report/results.md')
print('Figures', output / 'report/pdf_figs')
